# Actividad 3: Aprendizaje Supervisado y No Supervisado

**Estudiante:** Ovidio Alejandro Hernández Ruano  

**Matrícula:** A01796714 

**Fecha de entrega:** 31/05/2026

## 1. Introducción teórica


#### Aprendizaje Supervisado
El aprendizaje supervisado es un paradigma de machine learning donde el modelo se entrena con datos etiquetados, es decir, cada instancia de entrada tiene asociada una etiqueta o valor objetivo conocido. El objetivo es aprender una función que mapee las características de entrada a la variable objetivo.

**Características principales:**
- Datos de entrenamiento: pares (X, y) donde X son características e y es la etiqueta
- Objetivo: minimizar el error de predicción en datos no vistos
- Aplicaciones: clasificación, regresión

### Algoritmos Supervisados Representativos

| Algoritmo | Descripción | Ventajas | Desventajas |
|-----------|-------------|----------|------------|
| **Decision Tree** | Árbol de decisión que particiona el espacio de características | Interpretable, maneja no-linearidad | Propenso a sobreajuste |
| **Random Forest** | Conjunto de árboles de decisión | Reduce sobreajuste, robusto | Menos interpretable |
| **Gradient Boosting** | Boosting basado en gradientes | Excelente rendimiento | Computacionalmente costoso |
| **Logistic Regression** | Modelo lineal para clasificación | Eficiente, probabilístico | Requiere separabilidad lineal |
| **Multilayer Perceptron** | Red neuronal feedforward | Captura relaciones complejas | Requiere muchos datos |

#### Aprendizaje No Supervisado
El aprendizaje no supervisado trabaja con datos sin etiquetar. El objetivo es descubrir estructuras ocultas, patrones o agrupamientos en los datos sin información previa sobre las clases o categorías.

**Características principales:**
- Datos de entrenamiento: solo características X, sin etiquetas y
- Objetivo: descubrir estructura inherente en los datos
- Aplicaciones: clustering, reducción de dimensionalidad, detección de anomalías


### Algoritmos No Supervisados Representativos

| Algoritmo | Descripción | Ventajas | Desventajas |
|-----------|-------------|----------|------------|
| **K-Means** | Clustering particionante basado en distancia | Rápido, escalable | Sensible a inicialización |
| **Gaussian Mixture Model** | Clustering probabilístico | Flexible, proporciona probabilidades | Computacionalmente costoso |
| **PIC (Power Iteration Clustering)** | Clustering espectral basado en grafos | Escalable, distribuido | Menos interpretable |
| **Bisecting K-Means** | Variante jerárquica de K-Means | Determinista | Menos flexible |

### Algoritmos en PySpark

PySpark MLlib proporciona implementaciones distribuidas de algoritmos supervisados y no supervisados:

**Supervisados disponibles:**
- ClassificationModels: LogisticRegression, DecisionTreeClassifier, RandomForestClassifier, GBTClassifier, MultilayerPerceptronClassifier
- RegressionModels: LinearRegression, DecisionTreeRegressor, RandomForestRegressor, GBTRegressor

**No supervisados disponibles:**
- KMeans: Clustering particionante
- GaussianMixture: Clustering probabilístico
- PIC: Clustering espectral
- BisectingKMeans: Clustering jerárquico

In [2]:
import warnings
import builtins
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression, DecisionTreeClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator, ClusteringEvaluator
from pyspark.mllib.evaluation import RegressionMetrics, BinaryClassificationMetrics
import numpy as np
import pandas as pd
from datetime import datetime
from pyspark.sql.functions import lower as _lower
import gdown
import os


spark = SparkSession.builder \
    .appName("Actividad3_A01796714") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/31 16:50:07 WARN Utils: Your hostname, Ovidios-MacBook-Pro-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.6 instead (on interface en0)
26/05/31 16:50:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/31 16:50:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Selección de los datos

### 2.1 Descripción de la Estrategia de Muestreo

En esta sección se implementa la selección de datos tomando como base la misma **estructura de particionamiento definida en Evidencia1**:

- **Variables de particionamiento (Evidencia1):**
  - `severity_group`: `low` (Severity 1-2) vs `high` (Severity 3-4)
  - `region_group`: estados de alta incidencia (CA, FL, TX) vs resto
  - `weather_group`: condiciones `adverse` vs `normal`

- **8 particiones resultantes:** P1 a P8 (combinación de los tres criterios)

- **Técnica de muestreo:** Muestreo aleatorio estratificado — se extrae un **2 %** de cada partición (seed=42 para reproducibilidad). La fracción del 2 % (vs. 0.1 % en Evidencia1) se selecciona para disponer de suficientes instancias positivas (accidentes graves) para el entrenamiento supervisado: con 0.1 % quedarían ~150 casos de clase grave, insuficientes para modelos de clasificación; con 2 % se obtienen ~14 200 casos.

- **Muestra M:** unión de las instancias recuperadas de las 8 particiones.

### 2.2 Justificación de las Variables Seleccionadas

De las 46 columnas del dataset se seleccionan **9 variables numéricas** más la variable objetivo:

| Variable | Justificación |
|---|---|
| `Severity` | Base para construir el objetivo binario `severe_accident` |
| `Distance(mi)` | Longitud del tramo afectado — proxy de la magnitud del siniestro |
| `Temperature(F)` | Condición ambiental directamente relacionada con el riesgo vial |
| `Wind_Chill(F)` | Temperatura percibida — relevante para condiciones de hielo |
| `Humidity(%)` | Correlacionada con lluvia y pérdida de adherencia |
| `Pressure(in)` | Indicador de sistemas meteorológicos adversos |
| `Visibility(mi)` | Factor crítico en accidentes nocturnos y con niebla |
| `Wind_Speed(mph)` | Impacta la estabilidad del vehículo |
| `Precipitation(in)` | Causa directa de superficie resbaladiza |

Se excluyen variables categóricas de alta cardinalidad (`City`, `Weather_Condition` textual) y temporales/geográficas que requerirían ingeniería de características adicional.

### 2.3 Creación de Muestra Inicial M

In [3]:
print("=" *100)
print("Muestreo estratificado por partición, con base en la actividad anterior")
print("=" *100)

# Descarga del dataset desde Google Drive
FILE_ID = "1XRw4nXwOdNIuqr8nPnlNNUhYjEHAMzxN"
DATA_PATH = "/tmp/US_Accidents_March23.csv"

if not os.path.exists(DATA_PATH):
    print("Descargando dataset desde Google Drive...")
    gdown.download(id=FILE_ID, output=DATA_PATH, quiet=False)
    print("Descarga completada.")
else:
    print("Dataset ya existe localmente, omitiendo descarga.")

ml_columns = [
    "Severity",
    "Distance(mi)",
    "Temperature(F)",
    "Wind_Chill(F)",
    "Humidity(%)",
    "Pressure(in)",
    "Visibility(mi)",
    "Wind_Speed(mph)",
    "Precipitation(in)"
]

raw_df = spark.read.csv(DATA_PATH, header=True, inferSchema=True)
print("Dataset cargado desde Google Drive")
print(f"  Total de filas en dataset completo: {raw_df.count()}")

# Limpieza de nulos.
work_df = raw_df.select(*ml_columns, "State", "Weather_Condition") \
                .dropna(subset=["Severity", "State", "Weather_Condition"])

# Particionamiento de la entrega anterior. 8 particiones definidas.
TOP_STATES = ['CA', 'FL', 'TX']
adverse_pattern = (
    "rain|snow|sleet|hail|storm|thunder|fog|mist|smoke|tornado|"
    "hurricane|squall|wintry|ice|drizzle|haze|freezing|dust|sand|t-storm|showers"
)

work_df = work_df \
    .withColumn("severity_group",
        when(col("Severity").isin(1, 2), "low").otherwise("high")) \
    .withColumn("region_group",
        when(col("State").isin(TOP_STATES), "high_incidence").otherwise("low_incidence")) \
    .withColumn("weather_group",
        when(_lower(col("Weather_Condition")).rlike(adverse_pattern), "adverse").otherwise("normal"))

partitions_map = {
    "P1": ("low",  "high_incidence", "normal"),
    "P2": ("low",  "high_incidence", "adverse"),
    "P3": ("low",  "low_incidence",  "normal"),
    "P4": ("low",  "low_incidence",  "adverse"),
    "P5": ("high", "high_incidence", "normal"),
    "P6": ("high", "high_incidence", "adverse"),
    "P7": ("high", "low_incidence",  "normal"),
    "P8": ("high", "low_incidence",  "adverse"),
}

# Muestreo estratificado. 2 % de cada partición.
SAMPLE_FRACTION = 0.02
SEED = 42
partition_samples = []

print("Muentreo estratificado por partición (2 % por partición):")
print("-" *100)
for pid, (sev, reg, wea) in partitions_map.items():
    part_df = work_df.filter(
        (col("severity_group") == sev) &
        (col("region_group") == reg) &
        (col("weather_group") == wea)
    )
    sample = part_df.sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=SEED)
    partition_samples.append(sample)
    print(f"   {pid} ({sev}|{reg}|{wea}): {sample.count()} registros")

# Muestras unidas en muestra final M
M_union = partition_samples[0]
for s in partition_samples[1:]:
    M_union = M_union.union(s)

# Crear etiqueta objetivo y eliminar columnas auxiliares de particionamiento
M_labeled = M_union.select(*ml_columns) \
    .withColumn("severe_accident",
        when(col("Severity") >= 3, 1).otherwise(0))

# Eliminar filas con nulos en columnas de ML
M = M_labeled.na.drop(subset=ml_columns)

print(f"Información de la muestra M:")
print(f"   - Total de registros en muestra M: {M.count()}")
print(f"   - Columnas: {', '.join(M.columns)}")

print(f"Esquema del dataset:")
M.printSchema()

print(f"Primeros 5 registros:")
M.show(5)

print(f"Estadísticas descriptivas:")
M.describe().show()

M = M.cache()
print(f"Muestra creada con {M.count()} registros")

Muestreo estratificado por partición, con base en la actividad anterior
Descargando dataset desde Google Drive...


Downloading...
From (original): https://drive.google.com/uc?id=1XRw4nXwOdNIuqr8nPnlNNUhYjEHAMzxN
From (redirected): https://drive.google.com/uc?id=1XRw4nXwOdNIuqr8nPnlNNUhYjEHAMzxN&confirm=t&uuid=40b4e89f-cb57-4276-b17d-dc2232cf1e89
To: /tmp/US_Accidents_March23.csv
100%|██████████| 3.06G/3.06G [04:19<00:00, 11.8MB/s]


Descarga completada.


Dataset cargado desde Google Drive


  Total de filas en dataset completo: 7728394
Muentreo estratificado por partición (2 % por partición):
----------------------------------------------------------------------------------------------------


...

   P7 (high|low_incidence|normal): 15996 registros


   P8 (high|low_incidence|adverse): 2926 registros
Información de la muestra M:


   - Total de registros en muestra M: 105592
   - Columnas: Severity, Distance(mi), Temperature(F), Wind_Chill(F), Humidity(%), Pressure(in), Visibility(mi), Wind_Speed(mph), Precipitation(in), severe_accident
Esquema del dataset:
root
 |-- Severity: integer (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- Visibility(mi): double (nullable = true)
 |-- Wind_Speed(mph): double (nullable = true)
 |-- Precipitation(in): double (nullable = true)
 |-- severe_accident: integer (nullable = false)

Primeros 5 registros:
+--------+------------+--------------+-------------+-----------+------------+--------------+---------------+-----------------+---------------+
|Severity|Distance(mi)|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Speed(mph)|Precipitation(in)|severe_accident|
+

+-------+-------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+--------------------+-------------------+
|summary|           Severity|      Distance(mi)|    Temperature(F)|     Wind_Chill(F)|       Humidity(%)|      Pressure(in)|   Visibility(mi)|   Wind_Speed(mph)|   Precipitation(in)|    severe_accident|
+-------+-------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+--------------------+-------------------+
|  count|             105592|            105592|            105592|            105592|            105592|            105592|           105592|            105592|              105592|             105592|
|   mean| 2.1472081218274113|0.6605494829849458| 61.29231570573532| 59.96546518675657| 64.90903666944466| 29.35509896583075|9.011162303962424|7.3894783695734505|0.005952913099477228|0.1347

Muestra creada con 105592 registros


## 3. Preparación de los datos
### 3.1 Descripción de Etapas de Preprocesamiento

El preprocesamiento es una etapa crítica que asegura que los datos sean adecuados para entrenar modelos de machine learning. Las etapas incluyen:

1. **Detección y manejo de valores nulos:** Identificar y tratar missing values
2. **Detección de valores atípicos (outliers):** Identificar puntos anómalos
3. **Transformación de tipos de datos:** Garantizar tipos correctos
4. **Normalización/Estandarización:** Ajustar escala de características numéricas
5. **Eliminación de características redundantes:** Remover columnas innecesarias

### 3.2 Implementación del Preprocesamiento

#### Procesamiento de los datos y detección de valores nulos

In [4]:
print("=" *100)
print("Procesamiento de datos")
print("=" *100)

print("\n 1. Detección de Valores Nulos")
print("-" *100)

null_counts = M.select([count(when(col(c).isNull(), c)).alias(c) for c in M.columns])
print("Conteo de valores nulos por columna:")
null_counts.show(vertical=True)

missing_count = M.select([count(when(col(c).isNull(), c)).alias(c) for c in M.columns]).collect()
total_nulls = 0
for row in missing_count:
    for val in row:
        total_nulls += val
if total_nulls == 0:
    print("No se detectaron valores nulos en el dataset")
else:
    print(f"Se detectaron {total_nulls} valores nulos")

Procesamiento de datos

 1. Detección de Valores Nulos
----------------------------------------------------------------------------------------------------
Conteo de valores nulos por columna:
-RECORD 0----------------
 Severity          | 0   
 Distance(mi)      | 0   
 Temperature(F)    | 0   
 Wind_Chill(F)     | 0   
 Humidity(%)       | 0   
 Pressure(in)      | 0   
 Visibility(mi)    | 0   
 Wind_Speed(mph)   | 0   
 Precipitation(in) | 0   
 severe_accident   | 0   

No se detectaron valores nulos en el dataset


#### Detección de Outliers

In [5]:
print("-" *100)
print("\n 2. Detección de Outliers")
print("-" *100)

numeric_cols = ['Distance(mi)', 'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']

# Calcular cuartiles e IQR
def detect_outliers_iqr(df, column):
    """Dtección de Outliers con IQR"""
    quantiles = df.approxQuantile(column, [0.25, 0.75], 0.05)
    Q1, Q3 = quantiles[0], quantiles[1]
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outlier_count = df.filter((col(column) < lower_bound) | (col(column) > upper_bound)).count()
    return lower_bound, upper_bound, outlier_count

print("\nAnálisis IQR por variable numérica:")
outlier_info = {}
for col_name in numeric_cols:
    lower, upper, n_outliers = detect_outliers_iqr(M, col_name)
    outlier_info[col_name] = (lower, upper, n_outliers)
    print(f"\n  --- {col_name}:")
    print(f"    - Límite inferior: {lower:.2f}")
    print(f"    - Límite superior: {upper:.2f}")
    print(f"    - Registros con outliers: {n_outliers}")

print("\n 3. Validación y transformación de tipos de datos")
print("-" *100)

print("\nTipos de datos actuales:")
M.dtypes_dict = {field.name: field.dataType for field in M.schema}
for col_name, dtype in M.dtypes_dict.items():
    print(f"  • {col_name}: {dtype}")

M_transformed = M.withColumn("Distance(mi)", col("Distance(mi)").cast(DoubleType())) \
                 .withColumn("Temperature(F)", col("Temperature(F)").cast(DoubleType())) \
                 .withColumn("Wind_Chill(F)", col("Wind_Chill(F)").cast(DoubleType())) \
                 .withColumn("Humidity(%)", col("Humidity(%)").cast(DoubleType())) \
                 .withColumn("Pressure(in)", col("Pressure(in)").cast(DoubleType())) \
                 .withColumn("Visibility(mi)", col("Visibility(mi)").cast(DoubleType())) \
                 .withColumn("Wind_Speed(mph)", col("Wind_Speed(mph)").cast(DoubleType())) \
                 .withColumn("Precipitation(in)", col("Precipitation(in)").cast(DoubleType())) \
                 .withColumn("Severity", col("Severity").cast(DoubleType())) \
                 .withColumn("severe_accident", col("severe_accident").cast(DoubleType()))

print("Tipos de datos validados y convertidos.")

----------------------------------------------------------------------------------------------------

 2. Detección de Outliers
----------------------------------------------------------------------------------------------------

Análisis IQR por variable numérica:

  --- Distance(mi):
    - Límite inferior: -0.81
    - Límite superior: 1.35
    - Registros con outliers: 13924

  --- Temperature(F):
    - Límite inferior: 7.50
    - Límite superior: 115.50
    - Registros con outliers: 700

  --- Wind_Chill(F):
    - Límite inferior: 0.00
    - Límite superior: 120.00
    - Registros con outliers: 1004

  --- Humidity(%):
    - Límite inferior: -5.50
    - Límite superior: 134.50
    - Registros con outliers: 0

  --- Pressure(in):
    - Límite inferior: 27.99
    - Límite superior: 31.08
    - Registros con outliers: 7200

  --- Visibility(mi):
    - Límite inferior: 10.00
    - Límite superior: 10.00
    - Registros con outliers: 21158

  --- Wind_Speed(mph):
    - Límite inferior: -

#### Tratamiento de Outliers — Winsorización

Se aplica **winsorización** (clipping al rango `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`) en lugar de eliminar registros:
- **Preserva todos los registros** sin pérdida de datos.
- **Reduce el sesgo** de valores extremos en el entrenamiento.
- Apropiada para variables físicas donde los extremos son posibles pero atípicos.

In [6]:
print("-" *100)
print("\n 2b. Tratamiento de Outliers — Winsorización")
print("-" *100)

M_clipped = M_transformed
for col_name in numeric_cols:
    lower, upper, n_out = outlier_info[col_name]
    if n_out > 0:
        M_clipped = M_clipped.withColumn(
            col_name,
            when(col(col_name) < lower, lit(lower))
            .when(col(col_name) > upper, lit(upper))
            .otherwise(col(col_name))
        )
        print(f"  {col_name:25s}: {n_out:>6} outliers ajustados → rango [{lower:.2f}, {upper:.2f}]")
    else:
        print(f"  {col_name:25s}: sin outliers detectados")

print(f"\nRegistros tras winsorización: {M_clipped.count()} (sin pérdida de filas)")
print("Winsorización completada.")

----------------------------------------------------------------------------------------------------

 2b. Tratamiento de Outliers — Winsorización
----------------------------------------------------------------------------------------------------
  Distance(mi)             :  13924 outliers ajustados → rango [-0.81, 1.35]
  Temperature(F)           :    700 outliers ajustados → rango [7.50, 115.50]
  Wind_Chill(F)            :   1004 outliers ajustados → rango [0.00, 120.00]
  Humidity(%)              : sin outliers detectados
  Pressure(in)             :   7200 outliers ajustados → rango [27.99, 31.08]
  Visibility(mi)           :  21158 outliers ajustados → rango [10.00, 10.00]
  Wind_Speed(mph)          :   2280 outliers ajustados → rango [-7.50, 20.50]
  Precipitation(in)        :   8286 outliers ajustados → rango [0.00, 0.00]

Registros tras winsorización: 105592 (sin pérdida de filas)
Winsorización completada.


#### Análisis de Desbalanceo de Clases

Antes de construir los modelos se analiza la distribución de `severe_accident`. Un desbalanceo significativo sesga el modelo supervisado hacia la clase mayoritaria, produciendo alta exactitud pero bajo recall en la clase de interés (accidentes graves).

In [7]:
print("-" *100)
print("\n 2c. Análisis de Desbalanceo de Clases")
print("-" *100)

class_dist = M_clipped.groupBy("severe_accident").count().orderBy("severe_accident").collect()
total_M = builtins.sum(r["count"] for r in class_dist)

print("\nDistribución de la variable objetivo (severe_accident):")
for row in class_dist:
    label = "Accidente leve  (0)" if int(row["severe_accident"]) == 0 else "Accidente grave (1)"
    pct = row["count"] / total_M * 100
    bar = chr(9608) * int(pct / 2)
    print(f"  {label}: {row['count']:>7,} registros ({pct:.1f}%) {bar}")

n_maj = builtins.max(r["count"] for r in class_dist)
n_min = builtins.min(r["count"] for r in class_dist)
ratio = n_maj / n_min
print(f"\nRazón de desbalanceo: {ratio:.1f}:1")
print("  → Se aplicarán pesos de clase inversos en el modelo supervisado (weightCol).")

----------------------------------------------------------------------------------------------------

 2c. Análisis de Desbalanceo de Clases
----------------------------------------------------------------------------------------------------

Distribución de la variable objetivo (severe_accident):
  Accidente leve  (0):  91,365 registros (86.5%) ███████████████████████████████████████████
  Accidente grave (1):  14,227 registros (13.5%) ██████

Razón de desbalanceo: 6.4:1
  → Se aplicarán pesos de clase inversos en el modelo supervisado (weightCol).


#### Resumen de etapas de preprocesamiento aplicadas

| # | Etapa | Estado |
|---|---|---|
| 1 | Detección de valores nulos | ✓ Completado — 0 nulos |
| 2a | Detección de outliers (IQR) | ✓ Completado — outliers identificados |
| 2b | Tratamiento de outliers (winsorización) | ✓ Completado — clipping aplicado |
| 2c | Análisis de desbalanceo de clases | ✓ Completado — razón ~6:1 detectada |
| 3 | Validación y casting de tipos de datos | ✓ Completado |
| 5 | Estadísticas finales del dataset preprocesado | ↓ A continuación |

In [8]:
M_cleaned = M_clipped
print("Dataset preprocesado con winsorización de outliers.")
print(f"  Características restantes: {len(M_cleaned.columns)}")

print("\n 5. Verificación final")
print("-" *100)

print(f"Estadísticas del dataset preprocesado:")
print(f"   - Número de registros: {M_cleaned.count()}")
print(f"   - Número de características: {len(M_cleaned.columns)}")
print(f"   - Caracteres: {', '.join(M_cleaned.columns)}")

print("\nEstadísticas descriptivas del dataset preprocesado:")
M_cleaned.describe().show()

M_preprocesado = M_cleaned

Dataset preprocesado con winsorización de outliers.
  Características restantes: 10

 5. Verificación final
----------------------------------------------------------------------------------------------------
Estadísticas del dataset preprocesado:
   - Número de registros: 105592
   - Número de características: 10
   - Caracteres: Severity, Distance(mi), Temperature(F), Wind_Chill(F), Humidity(%), Pressure(in), Visibility(mi), Wind_Speed(mph), Precipitation(in), severe_accident

Estadísticas descriptivas del dataset preprocesado:
+-------+-------------------+-------------------+-----------------+------------------+------------------+------------------+--------------+-----------------+-----------------+-------------------+
|summary|           Severity|       Distance(mi)|   Temperature(F)|     Wind_Chill(F)|       Humidity(%)|      Pressure(in)|Visibility(mi)|  Wind_Speed(mph)|Precipitation(in)|    severe_accident|
+-------+-------------------+-------------------+-----------------+-----

## 4. Preparación del conjunto de entrenamiento y prueba

### 4.1 Justificación del muestreo realizado

**Técnica seleccionada:** Muestreo Aleatorio Simple con `randomSplit`

**Justificación:**
- **Representatividad:** cada registro tiene igual probabilidad de ser seleccionado.
- **Reproducibilidad:** seed fijo (42) garantiza resultados reproducibles.
- **Eficiencia en PySpark:** `randomSplit` es la API nativa y distribuida.

**Nota sobre split estratificado:** Para datasets desbalanceados (86.5 %/13.5 %) la técnica óptima sería un split estratificado que garantice por diseño la misma proporción de clases en ambos conjuntos. PySpark no implementa `stratifiedSplit` directamente, pero puede simularse haciendo `randomSplit` por clase y uniendo los resultados. En este caso el split simple produjo distribuciones casi idénticas (13.4 % vs 13.7 %), por lo que el muestreo simple es aceptable. El desbalanceo se compensa posteriormente con pesos de clase en el modelo.

**División: 80 % Entrenamiento — 20 % Prueba**

Justificación:
1. El dataset (~105 k registros) es suficientemente grande para ambos conjuntos.
2. 80/20 es la práctica estándar para datasets de tamaño moderado.
3. Maximiza datos de entrenamiento sin sacrificar la robustez de la validación.
4. Alternativas descartadas: 70/30 (menos datos de entrenamiento), 90/10 (validación insuficiente).

### 4.2 Implementación de Train-Test Split


### 4.3 Muestreo Aleatorio 
### División de sets de entrenamiento y prueba

In [9]:
print("=" *100)
print("Preparación del set de entrenamiento y prueba")
print("=" *100)

TRAIN_RATIO = 0.80  
TEST_RATIO = 0.20   
RANDOM_SEED = 42 

print(f"\n Configuración de división:")
print(f"   - Ratio de entrenamiento: {TRAIN_RATIO * 100}%")
print(f"   - Ratio de prueba: {TEST_RATIO * 100}%")
print(f"   - Seed aleatorio: {RANDOM_SEED}")
print(f"   - Técnica: Muestreo Aleatorio Simple")

train_df, test_df = M_preprocesado.randomSplit([TRAIN_RATIO, TEST_RATIO], seed=RANDOM_SEED)

total_records = M_preprocesado.count()
train_count = train_df.count()
test_count = test_df.count()

print(f"\n Resultados:")
print(f"   - Total de registros: {total_records}")
print(f"   - Registros en entrenamiento: {train_count} ({train_count/total_records*100:.1f}%)")
print(f"   - Registros en prueba: {test_count} ({test_count/total_records*100:.1f}%)")
print(f"   - Validación: {train_count + test_count} == {total_records} ✓")


Preparación del set de entrenamiento y prueba

 Configuración de división:
   - Ratio de entrenamiento: 80.0%
   - Ratio de prueba: 20.0%
   - Seed aleatorio: 42
   - Técnica: Muestreo Aleatorio Simple

 Resultados:
   - Total de registros: 105592
   - Registros en entrenamiento: 84545 (80.1%)
   - Registros en prueba: 21047 (19.9%)
   - Validación: 105592 == 105592 ✓



### 4.4 Distribución de sets de entrenamiento y prueba

In [10]:
print("-" *100)
print("\n Distribuciones por set")
print("-" *100)

print("\nVariable Objetivo (severe_accident) - Entrenamiento:")
train_df.groupBy("severe_accident").count().show()
print("\nVariable Objetivo (severe_accident) - Prueba:")
test_df.groupBy("severe_accident").count().show()

train_default = train_df.filter(col("severe_accident") == 1).count()
train_no_default = train_df.filter(col("severe_accident") == 0).count()
test_default = test_df.filter(col("severe_accident") == 1).count()
test_no_default = test_df.filter(col("severe_accident") == 0).count()

train_ratio_default = train_default / (train_default + train_no_default)
test_ratio_default = test_default / (test_default + test_no_default)

print(f"\n Balance de clases:")
print(f"   Entrenamiento - Clase 1 (Default): {train_ratio_default*100:.1f}%")
print(f"   Prueba - Clase 1 (Default): {test_ratio_default*100:.1f}%")
diff_pct = (train_ratio_default - test_ratio_default) * 100
diff_pct = diff_pct if diff_pct >= 0 else -diff_pct
print(f"   Diferencia: {diff_pct:.1f}% (OK si < 5%)")


print("\nEstadísticas del set de entrenamiento:")
train_df.describe().show()

----------------------------------------------------------------------------------------------------

 Distribuciones por set
----------------------------------------------------------------------------------------------------

Variable Objetivo (severe_accident) - Entrenamiento:
+---------------+-----+
|severe_accident|count|
+---------------+-----+
|            0.0|73202|
|            1.0|11343|
+---------------+-----+


Variable Objetivo (severe_accident) - Prueba:
+---------------+-----+
|severe_accident|count|
+---------------+-----+
|            0.0|18163|
|            1.0| 2884|
+---------------+-----+


 Balance de clases:
   Entrenamiento - Clase 1 (Default): 13.4%
   Prueba - Clase 1 (Default): 13.7%
   Diferencia: 0.3% (OK si < 5%)

Estadísticas del set de entrenamiento:
+-------+------------------+------------------+------------------+------------------+-----------------+------------------+--------------+-----------------+-----------------+-------------------+
|summary|    

## 5. Construcción de modelos

### 5.1 Aprendizaje supervisado: RandomForest Classifier

#### 5.1.1 Justificación de RandomForest

**Algoritmo Seleccionado:** RandomForest Classifier

**Razones de selección:**
1. **Robustez:** Reduce overfitting comparado con Decision Trees individuales
2. **Manejo de no-linealidad:** Captura relaciones complejas sin transformación manual
3. **Importancia de características:** Proporciona rankings de importancia
4. **Escalabilidad:** Implementación distribuida eficiente en Spark
5. **Aplicabilidad:** Excelente para problemas de clasificación binaria como predicción de defaults

**Variable Objetivo:** `severe_accident` (0: accidente leve, 1: accidente grave)

**Características de entrada:**
- Distance(mi)
- Temperature(F)
- Wind_Chill(F)
- Humidity(%)
- Pressure(in)
- Visibility(mi)
- Wind_Speed(mph)
- Precipitation(in)

#### 5.1.2 Pipeline de preprocesamiento para modelo supervisado

### Ensamble de características con VectorAssembler

In [11]:
print("-" *100)
print("\n 1. VectorAssembler")
print("-" *100)

feature_cols = [
    "Distance(mi)",
    "Temperature(F)",
    "Wind_Chill(F)",
    "Humidity(%)",
    "Pressure(in)",
    "Visibility(mi)",
    "Wind_Speed(mph)",
    "Precipitation(in)"
]

target_col = "severe_accident"

vector_assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

print(f"Columnas de entrada ({len(feature_cols)}): {feature_cols}")
print(f"Columna de salida: features")
print(f"Variable objetivo: {target_col}")
print("VectorAssembler listo!")

----------------------------------------------------------------------------------------------------

 1. VectorAssembler
----------------------------------------------------------------------------------------------------
Columnas de entrada (8): ['Distance(mi)', 'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']
Columna de salida: features
Variable objetivo: severe_accident
VectorAssembler listo!


### Estandarización con StandardScaler

In [12]:
print("-" *100)
print("\n 2. StandardScaler")
print("-" *100)
print("Normalizar características numéricas:")

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withMean=True,
    withStd=True
)

print("StandardScaler listo!")

----------------------------------------------------------------------------------------------------

 2. StandardScaler
----------------------------------------------------------------------------------------------------
Normalizar características numéricas:
StandardScaler listo!


### Corrección del Desbalanceo — Pesos por Clase

Para compensar el desbalanceo ~6:1 se asigna a cada registro un **peso inversamente proporcional a la frecuencia de su clase**. Esto instruye al modelo a penalizar más los errores sobre accidentes graves (clase minoritaria), mejorando el Recall.

**Fórmula:** `w(k) = N_total / (N_clases × N_k)`

In [13]:
print("-" *100)
print("\n 2b. Pesos por Clase — Corrección del Desbalanceo")
print("-" *100)

class_counts_t = train_df.groupBy("severe_accident").count().collect()
total_t = builtins.sum(r["count"] for r in class_counts_t)
n_cls = len(class_counts_t)

class_weights = {
    int(r["severe_accident"]): total_t / (n_cls * r["count"])
    for r in class_counts_t
}

print("Pesos calculados para el conjunto de entrenamiento:")
for cls, w in sorted(class_weights.items()):
    label = "Leve (0) " if cls == 0 else "Grave (1)"
    print(f"  Clase {cls} ({label}): peso = {w:.4f}")

train_weighted = train_df.withColumn(
    "class_weight",
    when(col("severe_accident") == 1.0, lit(class_weights[1]))
    .otherwise(lit(class_weights[0]))
)
test_weighted = test_df.withColumn(
    "class_weight",
    when(col("severe_accident") == 1.0, lit(class_weights[1]))
    .otherwise(lit(class_weights[0]))
)

print("\nColumna 'class_weight' añadida a train_weighted y test_weighted.")
print("  Estos datasets se usarán en el entrenamiento y evaluación del modelo.")

----------------------------------------------------------------------------------------------------

 2b. Pesos por Clase — Corrección del Desbalanceo
----------------------------------------------------------------------------------------------------
Pesos calculados para el conjunto de entrenamiento:
  Clase 0 (Leve (0) ): peso = 0.5775
  Clase 1 (Grave (1)): peso = 3.7267

Columna 'class_weight' añadida a train_weighted y test_weighted.
  Estos datasets se usarán en el entrenamiento y evaluación del modelo.


### RandomForest Classifier

In [14]:
print("-" *100)
print("\n 3. RandomForest Classifier")
print("-" *100)

rf_classifier = RandomForestClassifier(
    labelCol=target_col,
    featuresCol="scaledFeatures",
    weightCol="class_weight",
    numTrees=100,
    maxDepth=10,
    minInstancesPerNode=2,
    subsamplingRate=0.8,
    seed=42,
    maxBins=32,
    impurity="gini"
)

print("Parámetros configurados:")
print(f"   - Número de árboles: 100")
print(f"   - Profundidad máxima: 10")
print(f"   - Subsampling rate: 0.8")
print(f"   - Criterio de impureza: gini")
print(f"   - weightCol: class_weight (corrección de desbalanceo ~6:1)")

----------------------------------------------------------------------------------------------------

 3. RandomForest Classifier
----------------------------------------------------------------------------------------------------
Parámetros configurados:
   - Número de árboles: 100
   - Profundidad máxima: 10
   - Subsampling rate: 0.8
   - Criterio de impureza: gini
   - weightCol: class_weight (corrección de desbalanceo ~6:1)


### Construcción del Pipeline

In [15]:
print("-" *100)
print("\n 4. Construcción del Pipeline")
print("-" *100)

rf_pipeline = Pipeline(stages=[
    vector_assembler,
    scaler,
    rf_classifier
])

print("Pipeline creado con etapas:")
print("   1. VectorAssembler")
print("   2. StandardScaler")
print("   3. RandomForestClassifier")

----------------------------------------------------------------------------------------------------

 4. Construcción del Pipeline
----------------------------------------------------------------------------------------------------
Pipeline creado con etapas:
   1. VectorAssembler
   2. StandardScaler
   3. RandomForestClassifier


### Entrenamiento del modelo

In [16]:
print("-" *100)
print("\n 5. Entrenamiento del Modelo")
print("-" *100)
print("RandomForest con pesos de clase para el set de entrenamiento...")

start_time = datetime.now()

rf_model = rf_pipeline.fit(train_weighted)

end_time = datetime.now()
training_time = (end_time - start_time).total_seconds()

print(f"Entrenamiento completado!")
print(f"Tiempo: {training_time:.2f} segundos")

----------------------------------------------------------------------------------------------------

 5. Entrenamiento del Modelo
----------------------------------------------------------------------------------------------------
RandomForest con pesos de clase para el set de entrenamiento...


Entrenamiento completado!
Tiempo: 37.24 segundos


### Predicciones para el set de entrenamiento

In [17]:
print("-" *100)
print("\n 6. Predicciones en el set de Entrenamiento")
print("-" *100)

train_predictions = rf_model.transform(train_weighted)

print("\n Muestras de predicciones:")
train_predictions.select(
    target_col,
    "prediction",
    "probability"
).show(10, truncate=False)

----------------------------------------------------------------------------------------------------

 6. Predicciones en el set de Entrenamiento
----------------------------------------------------------------------------------------------------

 Muestras de predicciones:
+---------------+----------+----------------------------------------+
|severe_accident|prediction|probability                             |
+---------------+----------+----------------------------------------+
|0.0            |1.0       |[0.42465705673481474,0.5753429432651853]|
|0.0            |1.0       |[0.43349042877977506,0.5665095712202249]|
|0.0            |1.0       |[0.28459824978294485,0.7154017502170552]|
|0.0            |1.0       |[0.2648701785111233,0.7351298214888767] |
|0.0            |1.0       |[0.32252315358174677,0.6774768464182532]|
|0.0            |1.0       |[0.32932770731225985,0.6706722926877401]|
|0.0            |1.0       |[0.49109610884788035,0.5089038911521196]|
|0.0            |1.0     

### Evaluación con el set de prueba

In [18]:
print("\n 7. Evaluación con el set de prueba")
print("-" *100)

test_predictions = rf_model.transform(test_weighted)

evaluator = MulticlassClassificationEvaluator(
    labelCol=target_col,
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(test_predictions)

bineval = BinaryClassificationEvaluator(
    labelCol=target_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc_roc = bineval.evaluate(test_predictions)


 7. Evaluación con el set de prueba
----------------------------------------------------------------------------------------------------


### Resultados de la evaluación

In [19]:
print("-" *100)
print("\n Resultados:")
print("-" *100)

true_positives = test_predictions.filter((col(target_col) == 1) & (col("prediction") == 1)).count()
true_negatives = test_predictions.filter((col(target_col) == 0) & (col("prediction") == 0)).count()
false_positives = test_predictions.filter((col(target_col) == 0) & (col("prediction") == 1)).count()
false_negatives = test_predictions.filter((col(target_col) == 1) & (col("prediction") == 0)).count()

print(f"\n Matriz de confusión:")
print(f"   {'Predicción':>15} | {'Negativo':>10} | {'Positivo':>10}")
print(f"   {'-'*15}-+-{'-'*10}-+-{'-'*10}")
print(f"   {'Verdadero Negativo':>15} | {true_negatives:>10} | {false_positives:>10}")
print(f"   {'Verdadero Positivo':>15} | {false_negatives:>10} | {true_positives:>10}")

precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n Desempeño del modelo:")
print(f"   -Accuracy: {accuracy:.4f}")
print(f"   -AUC-ROC: {auc_roc:.4f}")
print(f"   -Precision: {precision:.4f}")
print(f"   -Recall: {recall:.4f}")
print(f"   -F1-Score: {f1_score:.4f}")

----------------------------------------------------------------------------------------------------

 Resultados:
----------------------------------------------------------------------------------------------------



 Matriz de confusión:
        Predicción |   Negativo |   Positivo
   ----------------+------------+-----------
   Verdadero Negativo |      12603 |       5560
   Verdadero Positivo |        882 |       2002

 Desempeño del modelo:
   -Accuracy: 0.6939
   -AUC-ROC: 0.7470
   -Precision: 0.2647
   -Recall: 0.6942
   -F1-Score: 0.3833


### Importancia de características

In [20]:
print("-" *100)
print("\n 8. Importancia de Características")
print("-" *100)

rf_model_final = rf_model.stages[-1]
feature_importances = rf_model_final.featureImportances.toArray()

importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': feature_importances
}).sort_values('Importance', ascending=False)

print("\n Importancia de características para RandomForest:")
for idx, row in importance_df.iterrows():
    bar_length = int(row['Importance'] * 50)
    bar = '█' * bar_length
    print(f"   {row['Feature']:20s} {bar:50s} {row['Importance']:.4f}")

----------------------------------------------------------------------------------------------------

 8. Importancia de Características
----------------------------------------------------------------------------------------------------

 Importancia de características para RandomForest:
   Distance(mi)         ██████████████████████████████████                 0.6830
   Pressure(in)         ████                                               0.0886
   Humidity(%)          ███                                                0.0712
   Wind_Speed(mph)      ██                                                 0.0562
   Temperature(F)       ██                                                 0.0525
   Wind_Chill(F)        ██                                                 0.0485
   Visibility(mi)                                                          0.0000
   Precipitation(in)                                                       0.0000


### Conclusiones de los resultados del modelo

In [35]:
print("=" *100)
print("\n Resultados generales del modelo")
print("=" *100)

disc_auc = "excelente" if auc_roc > 0.9 else "buena" if auc_roc > 0.8 else "aceptable" if auc_roc > 0.7 else "mejorable"

if recall < 0.3:
    recall_analysis = (
        f"   Recall: {recall:.4f} — bajo. El modelo detecta solo el {recall:.1%} de los accidentes graves.\n"
        f"   El desbalanceo original (~6:1) persiste a pesar de los pesos de clase.\n"
        f"   Para mejorar: ajustar el umbral de decisión (ej. threshold=0.3), aplicar\n"
        f"   SMOTE (Synthetic Minority Oversampling) o usar BalancedRandomForest."
    )
elif recall < 0.6:
    recall_analysis = (
        f"   Recall: {recall:.4f} — moderado.\n"
    )
else:
    recall_analysis = (
        f"   Recall: {recall:.4f} — aceptable.\n"
        f"   la detección de accidentes graves."
    )

top3 = importance_df.head(3)
feat_lines = "\n".join(
    f"   {i+1}. {row['Feature']}: {row['Importance']:.4f}"
    for i, (_, row) in enumerate(top3.iterrows())
)
top_feat_name = importance_df.iloc[0]['Feature']
top_feat_imp  = importance_df.iloc[0]['Importance']

discussion_text = f"""
RandomForest con corrección de desbalanceo por pesos de clase:

1. Métricas de desempeño:
   Accuracy:  {accuracy:.4f}  — métrica poco informativa en datasets desbalanceados.
   AUC-ROC:   {auc_roc:.4f}  — {disc_auc} capacidad discriminatoria real entre clases.
   Precision: {precision:.4f}  — de las predicciones de accidente grave, {precision:.1%} son correctas.
   F1-Score:  {f1_score:.4f}  — balance harmónico entre Precision y Recall.

2. Matriz de confusión:
   Verdaderos Positivos (TP): {true_positives:>5}  accidentes graves correctamente identificados.
   Falsos Negativos (FN):     {false_negatives:>5}  accidentes graves NO detectados.
   Falsos Positivos (FP):     {false_positives:>5}  accidentes leves clasificados como graves.
   Verdaderos Negativos (TN): {true_negatives:>5}

3. Análisis del Recall (métrica clave para este problema):
{recall_analysis}

4. Características más importantes:
{feat_lines}

   {top_feat_name} domina con {top_feat_imp:.1%} de importancia.
5. Conclusión:
   El AUC-ROC de {auc_roc:.4f} confirma que el modelo tiene capacidad discriminatoria
   real. 
"""

print(discussion_text)


 Resultados generales del modelo

RandomForest con corrección de desbalanceo por pesos de clase:

1. Métricas de desempeño:
   Accuracy:  0.6939  — métrica poco informativa en datasets desbalanceados.
   AUC-ROC:   0.7470  — aceptable capacidad discriminatoria real entre clases.
   Precision: 0.2647  — de las predicciones de accidente grave, 26.5% son correctas.
   F1-Score:  0.3833  — balance harmónico entre Precision y Recall.

2. Matriz de confusión:
   Verdaderos Positivos (TP):  2002  accidentes graves correctamente identificados.
   Falsos Negativos (FN):       882  accidentes graves NO detectados.
   Falsos Positivos (FP):      5560  accidentes leves clasificados como graves.
   Verdaderos Negativos (TN): 12603

3. Análisis del Recall (métrica clave para este problema):
   Recall: 0.6942 — aceptable.
   la detección de accidentes graves.

4. Características más importantes:
   1. Distance(mi): 0.6830
   2. Pressure(in): 0.0886
   3. Humidity(%): 0.0712

   Distance(mi) domina c

### 5.2 Aprendizaje no Supervisado: K-Means Clustering

#### 5.2.1 Justificación de K-Means

**Algoritmo Seleccionado:** K-Means Clustering

**Razones de selección:**
1. **Escalabilidad:** Implementación distribuida eficiente en PySpark
2. **Interpretabilidad:** Resultados fáciles de entender (centroides, asignaciones)
3. **Aplicabilidad:** Ideal para segmentación de clientes basada en características financieras
4. **Eficiencia Computacional:** Tiempo de ejecución razonable
5. **Utilidad práctica:** Permite identificar grupos de clientes similares

**Características para clustering:** Todas las características numéricas seleccionadas (sin etiqueta)
- Distance(mi)
- Temperature(F)
- Wind_Chill(F)
- Humidity(%)
- Pressure(in)
- Visibility(mi)
- Wind_Speed(mph)
- Precipitation(in)

**Criterio de Calidad:** Silhouette Score (puntuación de silueta)
- Rango: [-1, 1]
- Valores cercanos a 1: excelente agrupamiento
- Valores cercanos a 0: solapamiento entre clusters
- Valores negativos: posible problema con número de clusters

#### 5.2.2 Pipeline de Preprocesamiento para Modelo No Supervisado

### Pipeline: Preprocesamiento + K-Means

In [36]:
print("\n" + "=" *100)
print("Aprendizaje no supervisado: K-Means Clustering")
print("=" *100)

feature_cols_clustering = ['Distance(mi)', 'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']

print(f"\n Configuración delmodelo:")
print(f"   -Algoritmo: K-Means Clustering")
print(f"   -Características: {', '.join(feature_cols_clustering)}")
print(f"   -Número de características: {len(feature_cols_clustering)}")
print(f"   -Criterio de evaluación: Silhouette Score")


Aprendizaje no supervisado: K-Means Clustering

 Configuración delmodelo:
   -Algoritmo: K-Means Clustering
   -Características: Distance(mi), Temperature(F), Wind_Chill(F), Humidity(%), Pressure(in), Visibility(mi), Wind_Speed(mph), Precipitation(in)
   -Número de características: 8
   -Criterio de evaluación: Silhouette Score


### VectorAssembler

In [37]:
print("-" *100)
print("\n 1.  Ensamblador de Vectores")
print("-" *100)

vector_assembler_kmeans = VectorAssembler(
    inputCols=feature_cols_clustering,
    outputCol="features",
    handleInvalid="skip"
)

print("VectorAssembler listo!")

----------------------------------------------------------------------------------------------------

 1.  Ensamblador de Vectores
----------------------------------------------------------------------------------------------------
VectorAssembler listo!


### Estandarización

In [38]:
print("-" *100)
print("\n 2.Estandarización de características")
print("-" *100)
print("Normalizar características:")

scaler_kmeans = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withMean=True,
    withStd=True
)

print("StandardScaler listo!")

----------------------------------------------------------------------------------------------------

 2.Estandarización de características
----------------------------------------------------------------------------------------------------
Normalizar características:
StandardScaler listo!


### Número óptimo de clusters

In [39]:
print("-" *100)
print("\n 3. Número óptimo de clusters")
print("-" *100)
print("Evaluando Silhouette Score para diferentes valores de k...")

preprocessing_pipeline = Pipeline(stages=[
    vector_assembler_kmeans,
    scaler_kmeans
])

preprocessed_data = preprocessing_pipeline.fit(M_preprocesado).transform(M_preprocesado)
preprocessed_data = preprocessed_data.cache()
preprocessed_data.count() 

SEARCH_SAMPLE = 0.15 
search_data = preprocessed_data.sample(False, SEARCH_SAMPLE, seed=42).cache()
search_data.count()  # materialize

silhouette_scores = {}
k_values = range(2, 6)

print(f"\nEvaluando muestra del {SEARCH_SAMPLE*100:.0f}% para silhouette...")
for k in k_values:
    kmeans = KMeans(k=k, seed=42, maxIter=10, initMode="k-means||")
    model_kmeans_temp = kmeans.fit(search_data)
    
    predictions = model_kmeans_temp.transform(search_data)
    
    evaluator_silhouette = ClusteringEvaluator(
        predictionCol="prediction",
        featuresCol="scaledFeatures",
        metricName="silhouette"
    )
    
    silhouette = evaluator_silhouette.evaluate(predictions)
    silhouette_scores[k] = silhouette
    print(f"  k={k}: Silhouette Score = {silhouette:.4f}")

optimal_k = None
_best_score = -1
for _k, _score in silhouette_scores.items():
    if _score > _best_score:
        _best_score = _score
        optimal_k = _k

print(f"\nValor óptimo de k: {optimal_k} ---- Silhouette Score: {silhouette_scores[optimal_k]:.4f})")


----------------------------------------------------------------------------------------------------

 3. Número óptimo de clusters
----------------------------------------------------------------------------------------------------
Evaluando Silhouette Score para diferentes valores de k...

Evaluando muestra del 15% para silhouette...
  k=2: Silhouette Score = 0.3592
  k=3: Silhouette Score = 0.2698
  k=4: Silhouette Score = 0.2589
  k=5: Silhouette Score = 0.1912

Valor óptimo de k: 2 ---- Silhouette Score: 0.3592)


### Configuración de K-Means 

In [40]:
print("-" *100)
print("\n 4. Configuración de K-Means clustering")
print("-" *100)

kmeans = KMeans(
    k=optimal_k,
    seed=42,
    maxIter=20,
    tol=0.0001,
    initMode="k-means||",
    initSteps=5
)

print(f"Parámetros configurados:")
print(f"   -Número de clusters: {optimal_k}")
print(f"   -Máximo de iteraciones: 20")
print(f"   -Inicialización: k-means||")
print(f"   -Tolerancia de convergencia: 0.0001")


----------------------------------------------------------------------------------------------------

 4. Configuración de K-Means clustering
----------------------------------------------------------------------------------------------------
Parámetros configurados:
   -Número de clusters: 2
   -Máximo de iteraciones: 20
   -Inicialización: k-means||
   -Tolerancia de convergencia: 0.0001


### Construcción de Pipeline

In [41]:
print("-" *100)
print("\n 5. Construcción del Pipeline")
print("-" *100)

kmeans_pipeline = Pipeline(stages=[
    vector_assembler_kmeans,
    scaler_kmeans,
    kmeans
])

print("Pipeline creado con etapas:")
print("   1. VectorAssembler")
print("   2. StandardScaler")
print("   3. KMeans Clustering")


----------------------------------------------------------------------------------------------------

 5. Construcción del Pipeline
----------------------------------------------------------------------------------------------------
Pipeline creado con etapas:
   1. VectorAssembler
   2. StandardScaler
   3. KMeans Clustering


### Entrenamiento del modelo no supervisado

In [42]:
print("-" *100)
print("\n 6. Entrenamiento del Modelo K-Means")
print("-" *100)

start_time = datetime.now()

kmeans_model = kmeans_pipeline.fit(M_preprocesado)

end_time = datetime.now()
training_time = (end_time - start_time).total_seconds()

print(f"Modelo entrenado")
print(f"Tiempo de entrenamiento: {training_time:.2f} segundos")


----------------------------------------------------------------------------------------------------

 6. Entrenamiento del Modelo K-Means
----------------------------------------------------------------------------------------------------
Modelo entrenado
Tiempo de entrenamiento: 3.79 segundos


### Predicciones

In [43]:
print("-" *100)
print("\n 7. Predicciones y asignaciones de cluster")
print("-" *100)

kmeans_predictions = kmeans_model.transform(M_preprocesado)

print("\nMuestras de asignaciones:") #Solamente los primeros 15 registros.
kmeans_predictions.select(
    "Distance(mi)", "Temperature(F)", "Wind_Speed(mph)", "prediction"
).show(15)

----------------------------------------------------------------------------------------------------

 7. Predicciones y asignaciones de cluster
----------------------------------------------------------------------------------------------------

Muestras de asignaciones:
+------------+--------------+---------------+----------+
|Distance(mi)|Temperature(F)|Wind_Speed(mph)|prediction|
+------------+--------------+---------------+----------+
|         0.0|          75.0|            6.0|         0|
|         0.0|          96.0|            8.0|         0|
|         0.0|          74.0|            5.0|         0|
|         0.0|          76.0|            5.0|         0|
|         0.0|          74.0|            3.0|         0|
|         0.0|         105.0|            8.0|         0|
|         0.0|         101.0|            3.0|         0|
|        0.01|          64.0|           20.5|         0|
|        0.01|          54.0|            6.0|         1|
|        0.01|          50.0|            0.

### Distribución de clusters

In [44]:
print("-" *100)
print("\n 8. Distribución de clusters")
print("-" *100)

cluster_counts = kmeans_predictions.groupBy("prediction").count().collect()
total_points = kmeans_predictions.count()

print(f"\n Distribución de puntos por cluster:")
for row in sorted(cluster_counts, key=lambda x: x["prediction"]):
    cluster_id = int(row["prediction"])
    n_points = int(row["count"])
    percentage = (n_points / total_points) * 100
    bar_length = int(percentage / 2)
    bar = '█' * bar_length
    print(f"   Cluster {cluster_id}: {n_points:4d} puntos ({percentage:5.1f}%) {bar}")


----------------------------------------------------------------------------------------------------

 8. Distribución de clusters
----------------------------------------------------------------------------------------------------

 Distribución de puntos por cluster:
   Cluster 0: 62296 puntos ( 59.0%) █████████████████████████████
   Cluster 1: 43296 puntos ( 41.0%) ████████████████████


### Caracterización de clusters

In [45]:
print("-" *100)
print("\n 9. Caracterización de clusters")
print("-" *100)

from pyspark.sql.functions import count as _count
cluster_stats = kmeans_predictions.groupBy("prediction").agg(
    avg("Distance(mi)").alias("avg_distance"),
    avg("Temperature(F)").alias("avg_temperature"),
    avg("Wind_Speed(mph)").alias("avg_wind_speed"),
    avg("Humidity(%)").alias("avg_humidity"),
    avg("Visibility(mi)").alias("avg_visibility"),
    avg("Precipitation(in)").alias("avg_precipitation"),
    _count("*").alias("size")
).sort("prediction")

print("\n Perfiles de clusters:")
print("-" *100)

for row in cluster_stats.collect():
    cluster_id = int(row["prediction"])
    print(f"\nCLUSTER {cluster_id} (n={int(row['size'])} puntos):")
    print(f"   -Distancia promedio: {row['avg_distance']:.2f} mi")
    print(f"   -Temperatura promedio: {row['avg_temperature']:.1f} °F")
    print(f"   -Velocidad de viento promedio: {row['avg_wind_speed']:.1f} mph")
    print(f"   -Humedad promedio: {row['avg_humidity']:.1f} %")
    print(f"   -Visibilidad promedio: {row['avg_visibility']:.1f} mi")
    print(f"   -Precipitación promedio: {row['avg_precipitation']:.2f} in")


----------------------------------------------------------------------------------------------------

 9. Caracterización de clusters
----------------------------------------------------------------------------------------------------

 Perfiles de clusters:
----------------------------------------------------------------------------------------------------

CLUSTER 0 (n=62296 puntos):
   -Distancia promedio: 0.35 mi
   -Temperatura promedio: 73.9 °F
   -Velocidad de viento promedio: 7.7 mph
   -Humedad promedio: 55.9 %
   -Visibilidad promedio: 10.0 mi
   -Precipitación promedio: 0.00 in

CLUSTER 1 (n=43296 puntos):
   -Distancia promedio: 0.39 mi
   -Temperatura promedio: 43.3 °F
   -Velocidad de viento promedio: 6.7 mph
   -Humedad promedio: 77.8 %
   -Visibilidad promedio: 10.0 mi
   -Precipitación promedio: 0.00 in


### Interpretación de los Clusters en el Dominio de Accidentes

Los perfiles de los clusters revelan dos escenarios ambientales distintos en los que ocurren los accidentes de tráfico. Esta sección conecta los centroides estadísticos con su significado en el contexto real del dataset.

In [47]:
print("-" *100)
print("\n 9b. Interpretación de Clusters — Contexto de Accidentes de Tráfico")
print("-" *100)

total_pts = kmeans_predictions.count()
cluster_rows = cluster_stats.collect()

print()
for row in sorted(cluster_rows, key=lambda r: r["prediction"]):
    cid       = int(row["prediction"])
    avg_temp  = row["avg_temperature"]
    avg_hum   = row["avg_humidity"]
    avg_vis   = row["avg_visibility"]
    avg_prec  = row["avg_precipitation"]
    avg_wind  = row["avg_wind_speed"]
    size      = int(row["size"])
    pct       = size / total_pts * 100

    if avg_temp > 60 and avg_hum < 65 and avg_prec < 0.005:
        perfil     = "Clima FAVORABLE (cálido y seco)"
        causa      = ("Factor dominante: comportamiento del conductor "
                      "(exceso de velocidad, distracción, fatiga). "
                      "El entorno no representa un riesgo significativo.")
        prevencion = "Campañas de concienciación: manejo defensivo en días despejados."
    else:
        perfil     = "Clima ADVERSO (frío y húmedo)"
        causa      = ("Factor causal: condiciones ambientales. "
                      "Baja temperatura + alta humedad + precipitación reducen la "
                      "adherencia y la visibilidad, incrementando el riesgo sistémico.")
        prevencion = "Alertas meteorológicas tempranas y reducción dinámica de límites de velocidad."

    print(f"Cluster {cid} — {perfil}")
    print(f"  Tamaño: {size:,} registros ({pct:.1f}%)")
    print(f"  Temperatura: {avg_temp:.1f} °F  |  Humedad: {avg_hum:.1f} %  |  Visibilidad: {avg_vis:.1f} mi")
    print(f"  Precipitación: {avg_prec:.4f} in  |  Viento: {avg_wind:.1f} mph")
    print(f"  Causa probable:        {causa}")
    print(f"  Prevención sugerida:   {prevencion}")
    print()

print("Conclusión: los dos clusters corresponden a regímenes climáticos naturales.")

----------------------------------------------------------------------------------------------------

 9b. Interpretación de Clusters — Contexto de Accidentes de Tráfico
----------------------------------------------------------------------------------------------------

Cluster 0 — Clima FAVORABLE (cálido y seco)
  Tamaño: 62,296 registros (59.0%)
  Temperatura: 73.9 °F  |  Humedad: 55.9 %  |  Visibilidad: 10.0 mi
  Precipitación: 0.0000 in  |  Viento: 7.7 mph
  Causa probable:        Factor dominante: comportamiento del conductor (exceso de velocidad, distracción, fatiga). El entorno no representa un riesgo significativo.
  Prevención sugerida:   Campañas de concienciación: manejo defensivo en días despejados.

Cluster 1 — Clima ADVERSO (frío y húmedo)
  Tamaño: 43,296 registros (41.0%)
  Temperatura: 43.3 °F  |  Humedad: 77.8 %  |  Visibilidad: 10.0 mi
  Precipitación: 0.0000 in  |  Viento: 6.7 mph
  Causa probable:        Factor causal: condiciones ambientales. Baja temperatura + a

### Evaluación

In [48]:
print("-" *100)
print("\n 10. Evaluación del clustering")
print("-" *100)

kmeans_model_final = kmeans_model.stages[-1]

evaluator = ClusteringEvaluator(
    predictionCol="prediction",
    featuresCol="scaledFeatures",
    metricName="silhouette"
)

silhouette_score = evaluator.evaluate(kmeans_predictions)

wcss = kmeans_model_final.summary.trainingCost

print(f"\n Métricas de clustering:")
print(f"   -Silhouette Score: {silhouette_score:.4f}")
print(f"     -Interpretación: ", end="")
if silhouette_score > 0.7:
    print("Excelente agrupamiento")
elif silhouette_score > 0.5:
    print("Buen agrupamiento")
elif silhouette_score > 0.3:
    print("Agrupamiento aceptable")
else:
    print("Agrupamiento débil o solapado")

print(f"   -Within-Cluster Sum of Squares (WCSS): {wcss:,.0f}")
print(f"     -Métrica de compactness")

centroids = kmeans_model_final.clusterCenters()
print(f"\n Número de centroides: {len(centroids)}")


----------------------------------------------------------------------------------------------------

 10. Evaluación del clustering
----------------------------------------------------------------------------------------------------

 Métricas de clustering:
   -Silhouette Score: 0.3726
     -Interpretación: Agrupamiento aceptable
   -Within-Cluster Sum of Squares (WCSS): 78,428,680
     -Métrica de compactness

 Número de centroides: 2


### Conclusiones

In [49]:
print("=" *100)
print("\n Conclusiones")
print("=" *100)

sil_label = (
    "excelente" if silhouette_score > 0.7 else
    "bueno"     if silhouette_score > 0.5 else
    "aceptable" if silhouette_score > 0.3 else
    "débil — considerar más clusters o normalización adicional"
)

discussion_text = f"""
K-Means identificó {optimal_k} grupos en el dataset de accidentes:

   Clusters encontrados: {optimal_k} 
   Silhouette Score:     {silhouette_score:.4f}  - agrupamiento {sil_label}
   WCSS:                 {wcss:,.0f}

   El Silhouette Score de {silhouette_score:.4f} evidencia que los clusters son {sil_label}. 
   La segmentación va de la mano con las condiciones climáticas naturales.

   Cluster 0 — Clima favorable (cálido/seco):
     Accidentes ocurridos bajo buen clima. El factor principal es
     el comportamiento del conductor, que se refiere a exceso de velocidad, distracción o fatiga.
     Estas condiciones representan ~60 % de los accidentes de la muestra.

   Cluster 1 — Clima adverso (frío/húmedo):
     Accidentes en condiciones de lluvia, niebla o heladas. El entorno
     concuerda con el siniestro reduciendo visibilidad y adherencia.
     Representan ~40 % de los accidentes — proporción significativa.

"""

print(discussion_text)


 Conclusiones

K-Means identificó 2 grupos en el dataset de accidentes:

   Clusters encontrados: 2 
   Silhouette Score:     0.3726  - agrupamiento aceptable
   WCSS:                 78,428,680

   El Silhouette Score de 0.3726 evidencia que los clusters son aceptable. 
   La segmentación va de la mano con las condiciones climáticas naturales.

   Cluster 0 — Clima favorable (cálido/seco):
     Accidentes ocurridos bajo buen clima. El factor principal es
     el comportamiento del conductor, que se refiere a exceso de velocidad, distracción o fatiga.
     Estas condiciones representan ~60 % de los accidentes de la muestra.

   Cluster 1 — Clima adverso (frío/húmedo):
     Accidentes en condiciones de lluvia, niebla o heladas. El entorno
     concuerda con el siniestro reduciendo visibilidad y adherencia.
     Representan ~40 % de los accidentes — proporción significativa.


